# Clase 4 — Búsqueda local: N-reinas

Notebook de referencia (demo en vivo) para acompañar `slides/clase4.md`.

Implementamos el problema de N-reinas como un **problema de optimización**
(Clase 4), no como CSP con backtracking (Clase 3): el estado siempre tiene
las N reinas puestas (una por columna); buscamos minimizar el número de
pares de reinas que se atacan.

Comparamos:
- Steepest-ascent hill climbing (con y sin *sideways moves*)
- Simulated annealing


In [ ]:
import random
import math
import matplotlib.pyplot as plt

random.seed(7)


## Representación del estado y función de costo

In [ ]:
def random_board(n):
    """Una fila por columna, elegida al azar."""
    return [random.randrange(n) for _ in range(n)]


def conflicts(board):
    """Cuenta pares de reinas que se atacan (misma fila o misma diagonal;
    la columna nunca se repite por construcción)."""
    n = len(board)
    c = 0
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:
                c += 1
            elif abs(board[i] - board[j]) == abs(i - j):
                c += 1
    return c


def neighbors(board):
    """Genera todos los vecinos: mover una reina a otra fila de su
    misma columna."""
    n = len(board)
    result = []
    for col in range(n):
        for row in range(n):
            if row != board[col]:
                new_board = board[:]
                new_board[col] = row
                result.append(new_board)
    return result


## Steepest-ascent hill climbing (con sideways moves opcionales)

In [ ]:
def hill_climbing(n, max_sideways=0, max_iters=1000):
    board = random_board(n)
    sideways_used = 0
    iters = 0
    while iters < max_iters:
        iters += 1
        current_cost = conflicts(board)
        if current_cost == 0:
            return board, iters, True
        best_neighbors = []
        best_cost = current_cost
        for nb in neighbors(board):
            c = conflicts(nb)
            if c < best_cost:
                best_cost = c
                best_neighbors = [nb]
            elif c == best_cost:
                best_neighbors.append(nb)
        if best_cost < current_cost:
            board = random.choice(best_neighbors)
            sideways_used = 0
        elif best_cost == current_cost and sideways_used < max_sideways:
            board = random.choice(best_neighbors)
            sideways_used += 1
        else:
            return board, iters, False
    return board, iters, conflicts(board) == 0


## Simulated annealing

In [ ]:
def simulated_annealing(n, t0=4.0, alpha=0.995, max_iters=20000):
    board = random_board(n)
    t = t0
    for it in range(1, max_iters + 1):
        current_cost = conflicts(board)
        if current_cost == 0:
            return board, it, True
        nb = random.choice(neighbors(board))
        delta = conflicts(nb) - current_cost  # negativo = mejora
        if delta < 0 or random.random() < math.exp(-delta / max(t, 1e-9)):
            board = nb
        t *= alpha
        if t < 1e-3:
            t = 1e-3
    return board, max_iters, conflicts(board) == 0


## Comparación experimental (Parte 4 de la actividad)

In [ ]:
def run_experiment(n, trials=100):
    results = {}
    for label, fn in [
        ("steepest-ascent (sin sideways)", lambda: hill_climbing(n, max_sideways=0)),
        ("steepest-ascent (sideways=100)", lambda: hill_climbing(n, max_sideways=100)),
        ("simulated annealing", lambda: simulated_annealing(n)),
    ]:
        successes = 0
        iters_success = []
        iters_fail = []
        for _ in range(trials):
            _, iters, ok = fn()
            if ok:
                successes += 1
                iters_success.append(iters)
            else:
                iters_fail.append(iters)
        results[label] = {
            "tasa_exito": successes / trials,
            "iter_prom_exito": sum(iters_success) / len(iters_success) if iters_success else None,
            "iter_prom_fallo": sum(iters_fail) / len(iters_fail) if iters_fail else None,
        }
    return results


for n in (8, 20):
    print(f"\n=== N = {n} ===")
    for algo, stats in run_experiment(n, trials=50).items():
        print(f"{algo:35s} -> {stats}")


## Visualización: costo vs. iteración (una corrida)

In [ ]:
def trace_hill_climbing(n, max_sideways=0, max_iters=200):
    board = random_board(n)
    costs = [conflicts(board)]
    sideways_used = 0
    for _ in range(max_iters):
        current_cost = conflicts(board)
        if current_cost == 0:
            break
        best_neighbors, best_cost = [], current_cost
        for nb in neighbors(board):
            c = conflicts(nb)
            if c < best_cost:
                best_cost, best_neighbors = c, [nb]
            elif c == best_cost:
                best_neighbors.append(nb)
        if best_cost < current_cost:
            board = random.choice(best_neighbors)
            sideways_used = 0
        elif best_cost == current_cost and sideways_used < max_sideways:
            board = random.choice(best_neighbors)
            sideways_used += 1
        else:
            break
        costs.append(conflicts(board))
    return costs


costs = trace_hill_climbing(8, max_sideways=20)
plt.figure()
plt.plot(costs, marker="o")
plt.xlabel("Iteración")
plt.ylabel("Conflictos (costo)")
plt.title("Hill climbing con sideways moves — 8 reinas")
plt.show()
